In [1]:
print("hello")

hello


In [ ]:
from langchain.document_loaders import PyPDFLoader , DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
def loadPdfFilles(path):
    loader=DirectoryLoader(
        path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents=loader.load()
    return documents

In [ ]:
extractedData=loadPdfFilles("data")


In [ ]:
extractedData # list of pages 

In [ ]:
len(extractedData) #nbr of pages

turning pdfs into images and extract the text and turn it to json

In [ ]:
from pathlib import Path
from pdf2image import convert_from_path
from PIL import Image
import pytesseract
import json
import os

# ==========================================
# 1. BULLETPROOF PATHING
# ==========================================
current_path = Path.cwd()

# If running from inside the 'research' folder, go up one level
if current_path.name == "research":
    BASE_DIR = current_path.parent 
else:
    # Failsafe: Hardcoded absolute path
    BASE_DIR = Path(r"C:\Users\vivobook\Desktop\INPT\Me\Project\developpementProject\chatBotJuridiques-RAG-")

PDF_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "data" / "pages"
OCR_OUT_DIR = BASE_DIR / "artifacts" / "ocr"

# ==========================================
# 2. CONFIGURATION
# ==========================================
DPI = 150
POPPLER_PATH = r"C:\poppler\Library\bin"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
SLICE_HEIGHT = 80 

def extract_patches(img, slice_height=SLICE_HEIGHT):
    w, h = img.size
    patches = []
    coords = []
    for top in range(0, h, slice_height):
        box = (0, top, w, min(top + slice_height, h))
        patch = img.crop(box)
        patches.append(patch)
        coords.append(box)
    return patches, coords

def ocr_patches(patches):
    texts = []
    total = len(patches)
    for i, patch in enumerate(patches):
        # Progress indicator so you know it's not frozen
        print(f"      -> Extracting text from patch {i+1}/{total}...", end="\r")
        text = pytesseract.image_to_string(patch, lang='ara', config='--psm 6')
        texts.append(text.strip())
    print() # Move to the next line after finishing the patches
    return texts

def process_all_pdfs():
    print(f"--- Working Directory: {BASE_DIR} ---")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OCR_OUT_DIR.mkdir(parents=True, exist_ok=True)
    
    pdfs = list(PDF_DIR.glob("*.pdf"))
    if not pdfs:
        print(f"❌ Error: No PDFs found in {PDF_DIR}")
        return

    print(f"\n--- STEP 1: CONVERTING {len(pdfs)} PDFs TO IMAGES ---")
    for pdf_path in pdfs:
        doc_id = pdf_path.stem
        doc_out = OUT_DIR / doc_id
        doc_out.mkdir(parents=True, exist_ok=True)
        try:
            print(f"⏳ Converting {pdf_path.name}...")
            pages = convert_from_path(pdf_path, dpi=DPI, fmt="png", poppler_path=POPPLER_PATH)
            for i, page in enumerate(pages, start=1):
                page_path = doc_out / f"page_{i:03d}.png"
                if not page_path.exists():
                    page.save(page_path, "PNG")
            print(f"✅ {doc_id}: {len(pages)} pages processed.")
        except Exception as e:
            print(f"❌ Error converting {pdf_path.name}: {e}")

    print("\n--- STEP 2: EXTRACTING TEXT (OCR) ---")
    for doc_folder in OUT_DIR.iterdir():
        if not doc_folder.is_dir(): continue
        
        pages_images = sorted(doc_folder.glob("*.png"))
        print(f"\n📂 Processing Document: {doc_folder.name}")

        for img_path in pages_images:
            print(f"  📄 Reading {img_path.name}...")
            img = Image.open(img_path).convert("RGB")
            patches, coords = extract_patches(img)
            
            ocr_texts = ocr_patches(patches)

            ocr_data = {
                "source_pdf": doc_folder.name,
                "page_image": img_path.name,
                "data": []
            }
            
            for i, (bbox, text) in enumerate(zip(coords, ocr_texts)):
                if text and len(text) > 2: 
                    ocr_data["data"].append({
                        "patch_index": i,
                        "bbox": bbox,
                        "text": text
                    })

            json_name = f"{doc_folder.name}_{img_path.stem}_ocr.json"
            with open(OCR_OUT_DIR / json_name, "w", encoding="utf-8") as f:
                json.dump(ocr_data, f, ensure_ascii=False, indent=4)
            print(f"  💾 Saved JSON for {img_path.name}")

    print("\n🎉 All tasks completed successfully!")

process_all_pdfs()

Setup the embedding moudle 

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return a Multilingual embedding model optimized for Arabic.
    """
    model_name = "intfloat/multilingual-e5-base"
    
    model_kwargs = {'device': 'cpu'} 
    encode_kwargs = {'normalize_embeddings': True} 
    
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )
    return embeddings

embedding = download_embeddings()
print("✅ Embeddings model is ready for Arabic!")

In [ ]:
vector = embedding.embed_query("الإقتصاد")
print(len(vector))
print(vector)

connect to Pinecone DB

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
index_name = "juridique-rag-index"

pc = Pinecone(api_key=PINECONE_API_KEY)

print("✅ Setup is ready! Connected to Pinecone.")

In [ ]:
print(" Loading multilingual-e5-base model...")

embedding = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={'device': 'cpu'}, 
    encode_kwargs={'normalize_embeddings': True}
)

print("✅ Model loaded successfully!")

In [ ]:
if not pc.has_index(index_name):
    print(f"⏳ Creating a new index called '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=768,  
        metric="cosine", 
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("✅ Index created successfully!")
else:
    print(f"✅ Index '{index_name}' already exists. We will use it.")

In [ ]:
current_path = Path.cwd()
BASE_DIR = current_path.parent if current_path.name == "research" else current_path
OCR_DIR = BASE_DIR / "artifacts" / "ocr"

json_files = list(OCR_DIR.glob("*.json"))
print(f"📂 Found {len(json_files)} JSON files. Reading them...")

documents = []

for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        content = json.load(f)
        
        for item in content.get('data', []):
            text = item.get('text', '').strip()
            
            if text and len(text) > 2:
                e5_text = f"passage: {text}"
                
                metadata = {
                    "source": content.get('source_pdf', 'Unknown'),
                    "page": content.get('page_image', 'Unknown')
                }
                
                doc = Document(page_content=e5_text, metadata=metadata)
                documents.append(doc)

print(f" Prepared {len(documents)} chunks (paragraphs) ready for Pinecone.")

In [ ]:
print(" Uploading vectors to Pinecone... ")

docsearch = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name=index_name
)

print(" DONE! All your legal documents are now vectorized and stored in Pinecone!")

If you want to add  

In [ ]:
from langchain_core.documents import Document

dswith = Document(
    page_content="passage: سلام",
    metadata={"source": "me", "page": "none"} 
)

print("⏳ Adding manual document to Pinecone...")
print(docsearch.add_documents(documents=[dswith]))
print("✅ Document added successfully!")

retriever

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("اتفاقيات مع أطباء")
retrieved_docs